ENTSO-E Energy Data Analysis

Retrieves electricity market data from the ENTSO-E Transparency Platform via entsoe-py for 8 European bidding zones — AT, BE, BG, CZ, FR, GR, HU, ES — covering **April to November 2024** (UTC).
Authenticated via transparency.entsoe.eu (https://transparency.entsoe.eu/) API key. NoMatchingDataError is imported to gracefully handle missing data for certain country/period combinations.

In [2]:
from entsoe import EntsoePandasClient
from entsoe.exceptions import NoMatchingDataError
import pandas as pd

client = EntsoePandasClient(api_key="7308a46a-2e78-4bf2-a552-3295649e49e3")

countries = ['AT','BE','BG','CZ','FR','GR','HU','ES']

start = pd.Timestamp("2024-04-01", tz="UTC")
end   = pd.Timestamp("2024-11-01", tz="UTC")

Helper to get hourly solar production and solar price data for one country

Queries the ENTSO-E API for a given country and period, returning a clean hourly data.
Key handling steps: Resampling for both solar power production and prices, and an inner join to retain only timestamps where both series are available. Returns None on any missing or empty response.

In [1]:
def get_hourly_solar_and_price(client, country_code, start, end):
    df_gen = client.query_generation(country_code, start=start, end=end)
    if df_gen.empty:
        return None

    if isinstance(df_gen.columns, pd.MultiIndex):
        mask = df_gen.columns.get_level_values(1) == "Actual Aggregated"
        df_agg = df_gen.loc[:, mask].copy()
        df_agg.columns = df_agg.columns.get_level_values(0)
    else:
        df_agg = df_gen.copy()

    solar = df_agg.get("Solar")
    if solar is None:
        return None

    solar_hourly = solar.resample("1h").mean()

    price_code = country_code

    try:
        prices = client.query_day_ahead_prices(
            country_code=price_code, start=start, end=end
        )
    except NoMatchingDataError:
        return None
    except Exception:
        return None

    if prices.empty:
        return None

    prices_hourly = prices.resample("1h").mean()

    df = pd.DataFrame(index=solar_hourly.index)
    df["solar_MW"] = solar_hourly
    df = df.join(prices_hourly.to_frame(name="price_EUR_MWh"), how="inner")
    df = df.dropna()
    if df.empty:
        return None

    df["hour"] = df.index.hour
    return df
